In [ ]:
# OPTIONAL: Install
# pip install -qU langchain langchain-openai pydantic python-dotenv


## Tutorial: OpenAI/Gemini Function Calling Deep Dive (w/ JSON Schemas) 
We’ll define strict tool schemas with Pydantic and show structured outputs.


In [ ]:
from getpass import getpass
from langchain_openai import ChatOpenAI

# Enter your OpenRouter API key securely when prompted.
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

# OpenRouter provides an OpenAI-compatible API.
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# You can change this to any compatible OpenRouter model.
MODEL = "openai/gpt-4o-mini"

llm = ChatOpenAI(
    model=MODEL,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0,
    seed=42,
)

print("OpenRouter configured successfully.")
print("Model:", MODEL)
from typing import List
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain.tools import tool

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, seed=42)


### Step 1: Define a schema with Pydantic
We’ll create a `WeatherRequest` and expose a `get_weather` tool with a strict schema.


In [ ]:
class WeatherRequest(BaseModel):
    city: str = Field(..., description="City name, e.g., 'San Francisco'")
    unit: str = Field("celsius", description="Temperature unit: 'celsius' or 'fahrenheit'")

@tool("get_weather", args_schema=WeatherRequest, return_direct=True, description="Get the weather in a given city.")
def get_weather(city: str, unit: str = "celsius") -> str:

    # Demo: pretend fetch; return structured string
    sample = {"San Francisco": 18, "New York": 24, "London": 19}
    temp_c = sample.get(city, 20)
    if unit == "fahrenheit":
        temp = round((temp_c * 9/5) + 32)
        return f"{{\"city\": \"{city}\", \"temp\": {temp}, \"unit\": \"F\"}}"
    return f"{{\"city\": \"{city}\", \"temp\": {temp_c}, \"unit\": \"C\"}}"


### Step 2: Show the generated JSON schema
Pydantic provides JSON schema; most providers accept the same shape for function calling.


In [ ]:
from pprint import pprint

schema = WeatherRequest.model_json_schema()
pprint(schema)


### Step 3: Invoke the tool via LLM
We’ll simulate a tool-augmented chat: the model chooses tool + args, returns structured output.


In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.prompts import MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use tools when needed."),
    ("human", "What's the temperature in {city} in {unit}?"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

agent = create_tool_calling_agent(llm, tools=[get_weather], prompt=prompt)
executor = AgentExecutor(agent=agent, tools=[get_weather], verbose=True)

print(executor.invoke({"city": "San Francisco", "unit": "celsius"})["output"])
